# 实验二：自适应视频流与 QoE 优化（Adaptive Bitrate Streaming, ABR）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~12 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU）&nbsp;|&nbsp; **无需 GPU**

---

## 实验概述

自适应码率（Adaptive Bitrate Streaming, ABR）是视频流媒体系统的核心技术：当网络带宽随时间波动时，播放器需要动态选择合适的视频码率，在「画质」与「流畅度（不卡顿）」之间取得平衡。选错策略的代价很直接——码率太高会导致缓冲区耗尽、播放卡顿（Stall/Rebuffering）；码率太低则会浪费带宽、画质不必要地差。

本实验通过仿真一条随时间波动的无线网络带宽轨迹，对比三种 ABR 策略：固定高码率（贪心策略）、基于缓冲区门限的启发式规则（Heuristic），以及基于 Q-Learning（一种强化学习方法）预先学到的决策策略。你将看到相同的网络条件下，不同策略导致的卡顿次数、画质切换频率、以及综合体验质量（QoE）截然不同。

这正是 YouTube、Netflix、抖音等视频平台每天都在解决的问题：如何让播放器在你毫无感知的情况下，持续做出正确的码率决策。本实验对应课程中「自适应流媒体与 QoE 优化」部分的核心内容。

## 学习目标

完成本实验后，你应该能够：

1. 解释视频画质、缓冲区水位（Buffer Occupancy）与播放卡顿（Stall）三者之间的相互制约关系；
2. 比较固定码率、启发式规则、Q-Learning 三种 ABR 策略的决策逻辑与优劣；
3. 用自己的语言描述 QoE（用户体验质量）评分是如何综合平均画质、卡顿惩罚与码率切换惩罚计算得到的；
4. 读懂码率-缓冲区联合可视化图，识别卡顿发生的时刻与原因；
5. 说明为什么真实工业系统（如 YouTube）往往优先选择启发式规则而非为每个用户单独训练强化学习模型。

## 背景与基本原理

### 为什么固定高码率会导致卡顿？

播放器每次从服务器下载一小段视频（Chunk），下载速度取决于当前网络带宽。如果选择的码率超过了带宽能够支撑的下载速度，下载一段视频所需的时间就会超过这段视频的播放时长——缓冲区（Buffer，即已下载但还未播放的视频时长）会持续被消耗而得不到补充，最终耗尽归零，播放器被迫暂停等待缓冲，也就是**卡顿（Stall / Rebuffering）**。

反过来，如果码率选得远低于带宽上限，缓冲区会快速积累，画质却被不必要地压低——这就是「保守但浪费」的策略。

### QoE（用户体验质量）的三个组成部分

单纯追求「不卡顿」或「画质最高」都不是好策略，真正的用户体验质量（Quality of Experience, QoE）需要综合考虑：

| 组成部分 | 含义 | 对 QoE 的影响 |
|---------|------|--------------|
| 平均画质（Average Quality） | 播放期间选择的平均码率水平 | 越高越好，直接提升观感 |
| 卡顿惩罚（Stall Penalty） | 每次卡顿对体验的严重损害 | 每次卡顿都会大幅拉低 QoE，惩罚权重通常远高于画质提升的收益 |
| 切换惩罚（Switch Penalty） | 码率频繁跳变造成的视觉不适 | 忽高忽低的画质比稳定的中等画质体验更差 |

本实验采用简化公式：`QoE = 平均画质 - 卡顿次数 × 卡顿惩罚系数 - 切换次数 × 切换惩罚系数`，卡顿惩罚系数远大于切换惩罚系数，这与真实系统的设计理念一致——卡顿是用户最不能容忍的问题。

### 三种 ABR 策略的原理

- **① 固定高码率（Fixed High Bitrate）**：无论网络如何波动，始终选择最高档码率。这是一种「贪心」策略，完全忽视网络状态，画质表现最好，但一旦带宽不足就会频繁卡顿。
- **② 启发式规则（Heuristic，基于缓冲区门限）**：根据当前缓冲区水位做简单规则判断——缓冲区低于某阈值就降码率保安全，缓冲区充裕就升码率提画质。这是目前工业界最常用的思路（如 BOLA、缓冲区驱动算法），逻辑简单、无需训练、鲁棒性好。
- **③ Q-Learning 策略**：Q-Learning 是一种强化学习（Reinforcement Learning）方法，核心思想是维护一张「Q 表」，记录「在某个状态下采取某个动作能获得的长期收益」。训练过程可以概括为：观察当前状态（如缓冲区水位）→ 选择一个动作（码率档位）→ 获得奖励（QoE 增量）→ 根据奖励更新 Q 表中对应「状态-动作」的价值估计 → 长期迭代后，Q 表收敛到每个状态下的最优动作。本实验直接使用一张**预先训练好**的 Q 表，重点是观察其决策效果，而非训练过程本身。

## 实验设计

**带宽轨迹仿真**：模拟 100 秒的无线网络带宽波动，均值约 8 Mbps，叠加两种周期性波动和随机噪声，整体限制在 0.5–15 Mbps 之间，代表典型 WiFi 信号强度变化场景。

**码率档位**：三档可选码率——2 Mbps（低）、5 Mbps（中）、10 Mbps（高）。

**策略对比**：三种策略共享同一段带宽轨迹与同一套缓冲区仿真逻辑，唯一区别是「下一段视频选择哪个码率」的决策函数不同。

**Q 表设计**：本实验中的 Q 表是一个长度为 5 的数组，把缓冲区水位划分为 5 个区间（每 5 秒一档），每个区间对应一个预先学到的最优码率档位，模拟 Q-Learning 训练收敛后的最终策略（并保留 10% 概率随机探索，模拟真实强化学习的探索机制）。

**预期观察**：固定策略画质最高但卡顿最多；启发式策略卡顿较少但码率切换频繁；Q-Learning 策略在卡顿次数和切换次数上通常都能取得更好的平衡。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（无需 GPU） |
| 网络访问 | 不需要（Internet Off） |
| 主要依赖 | NumPy、Matplotlib（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验，无需上传数据或安装额外包。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

print("无外部依赖，纯 NumPy 实现 ✅")

## 步骤一：模拟带宽波动，观察码率选择面临的挑战

下面的代码生成一条 100 秒的仿真带宽轨迹，并画出三条虚线代表三种可选码率（2 / 5 / 10 Mbps）。

**观察重点**：带宽曲线在 2~12 Mbps 之间反复波动，多次跌破 10 Mbps（高码率线）甚至跌破 5 Mbps（中码率线）。如果播放器不做任何自适应，始终选择 10 Mbps 播放，会在带宽跌破 10 Mbps 的时段持续消耗缓冲区——这就是后续实验中「固定高码率」策略会频繁卡顿的根本原因。

> 思考：仅凭这条带宽曲线，你能大致判断哪些时间段最容易发生卡顿吗？

In [ ]:
# 模拟网络带宽轨迹（Mbps）：模拟 WiFi 信号波动
np.random.seed(42)
T = 200  # 200 个时间步（每步 0.5 秒，共 100 秒）
t = np.arange(T)

# 生成波动带宽：均值 8 Mbps，有周期性波动和随机噪声
bandwidth = 8 + 3 * np.sin(t * 0.05) + 2 * np.sin(t * 0.15) + np.random.normal(0, 1.5, T)
bandwidth = np.clip(bandwidth, 0.5, 15)  # 限制在 0.5~15 Mbps

plt.figure(figsize=(14, 4))
plt.plot(t * 0.5, bandwidth, 'b-', alpha=0.7, linewidth=1)
plt.fill_between(t * 0.5, 0, bandwidth, alpha=0.1, color='blue')
plt.axhline(y=2, color='red', linestyle='--', alpha=0.5, label='Low Bitrate (2 Mbps)')
plt.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Medium Bitrate (5 Mbps)')
plt.axhline(y=10, color='green', linestyle='--', alpha=0.5, label='High Bitrate (10 Mbps)')
plt.xlabel('Time (s)')
plt.ylabel('Bandwidth (Mbps)')
plt.title('Simulated Bandwidth Fluctuation (100s)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print("三条虚线代表三种可选视频码率。注意带宽在 2~12 Mbps 之间波动——")
print("如果始终选择高码率，网络差时会卡顿；选择低码率则不卡但画质差。")

## 步骤二：定义播放模拟器与三种码率决策策略

这部分代码分三块，共同构成实验的核心逻辑：

**① 播放模拟器 `simulate()`**：每个时间步（0.5 秒）计算「下载到的视频时长」并加入缓冲区，同时扣除已播放的时长；如果缓冲区归零则记为一次卡顿。模拟结束后，根据平均画质、卡顿次数、切换次数计算 QoE 分数。

**② 三种决策函数**：
- `strategy_fixed`：永远返回最高档（索引 2 = 10 Mbps），不参考任何网络状态；
- `strategy_heuristic`：读取当前缓冲区水位，缓冲区 < 3 秒降为低码率，> 10 秒升为高码率，5~10 秒之间选中码率，其余情况保持不变；
- `strategy_qlearning`：查表 `Q_table`（5 个缓冲区状态 → 最优动作），并保留 10% 概率随机探索。

**③ 运行三种策略并打印结果表**：三种策略共享同一条带宽轨迹，输出各自的平均画质（AvgQ）、卡顿次数（Stalls）、切换次数（Switches）、QoE 综合评分，便于直接对比。

> 思考：`Q_table = [0, 0, 1, 2, 2]` 这几个数字分别对应缓冲区从低到高的最优码率档位，你能从数值上看出它的决策倾向吗（偏保守还是偏激进）？

In [ ]:
# Video playback simulator
QUALITY_LEVELS = [2, 5, 10]  # Mbps: Low, Medium, High
CHUNK_DURATION = 2.0  # seconds per chunk
STEP_DURATION = 0.5  # seconds per simulation step

def simulate(bandwidth, strategy_fn, label):
    buffer = 5.0  # initial buffer (seconds)
    quality = 1  # start at medium
    
    history = {'buffer': [], 'quality': [], 'stall': []}
    total_stall = 0
    quality_changes = 0
    prev_quality = quality
    
    for bw in bandwidth:
        # Download: in STEP_DURATION seconds, we get bw*STEP_DURATION Mb of data
        # That data equals (bw*STEP_DURATION / bitrate) seconds of video
        data_downloaded = bw * STEP_DURATION  # Mb
        video_seconds = data_downloaded / QUALITY_LEVELS[quality]  # seconds of video
        buffer += video_seconds  # add to buffer
        buffer -= STEP_DURATION  # consume playback time
        
        # Stall detection
        stalled = buffer <= 0
        if stalled:
            total_stall += 1
            buffer = 0.0
        
        # Make decision for next chunk
        quality = strategy_fn(buffer, quality, bw, history)
        quality = max(0, min(2, quality))
        
        if quality != prev_quality:
            quality_changes += 1
            prev_quality = quality
        
        history['buffer'].append(buffer)
        history['quality'].append(QUALITY_LEVELS[quality])
        history['stall'].append(stalled)
    
    # QoE score
    avg_quality = np.mean(history['quality'])
    stall_penalty = total_stall * 3
    switch_penalty = quality_changes * 0.5
    qoe = avg_quality - stall_penalty - switch_penalty
    
    return history, dict(label=label, avg_quality=avg_quality,
                         stalls=total_stall, switches=quality_changes, qoe=qoe)

print("Simulator ready")


In [ ]:
# Strategy 1: Always highest bitrate
def strategy_fixed(buffer, quality, bw, history):
    return 2

# Strategy 2: Buffer-driven heuristic
def strategy_heuristic(buffer, quality, bw, history):
    if buffer < 3:
        return 0  # low buffer -> drop to low
    elif buffer > 10:
        return 2  # plenty of buffer -> go high
    elif buffer > 5:
        return 1  # moderate -> medium
    return quality

# Strategy 3: Pre-trained Q-table
Q_table = np.array([0, 0, 1, 2, 2])  # 5 buffer states -> optimal actions
def strategy_qlearning(buffer, quality, bw, history):
    state = min(int(buffer // 5), 4)
    if np.random.random() < 0.1:  # 10% exploration
        return np.random.randint(0, 3)
    return Q_table[state]

print("3 strategies defined")
print(f"Q-table: {Q_table}")
print("Buffer 0-5s -> Low | 5-10s -> Low | 10-15s -> Med | 15-20s -> High | 20s+ -> High")


In [ ]:
# Run all three strategies
hist_fixed, stats_fixed = simulate(bandwidth, strategy_fixed, 'Fixed')
hist_heur, stats_heur = simulate(bandwidth, strategy_heuristic, 'Heuristic')
hist_ql, stats_ql = simulate(bandwidth, strategy_qlearning, 'Q-Learning')

print("\n========== Results ==========")
for s in [stats_fixed, stats_heur, stats_ql]:
    print(f"{s['label']:12s} | AvgQ:{s['avg_quality']:5.1f} | Stalls:{s['stalls']:3d} | Switches:{s['switches']:3d} | QoE:{s['qoe']:6.1f}")


## 步骤三：可视化对比三种策略的实际表现

下方图表为三种策略分别绘制一张子图，包含两条曲线：

- **彩色阶梯线（左侧 y 轴）**：随时间变化选择的码率；
- **灰色曲线（右侧 y 轴，半透明背景）**：随时间变化的缓冲区水位；
- **红色竖线**：标记发生卡顿的时刻。

**观察重点**：对比红色竖线的数量与分布、彩色阶梯线的跳变频率、灰色缓冲区曲线贴近零的次数，直观感受三种策略在「卡顿」与「画质切换」之间的不同取舍。

In [ ]:
# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

histories = [hist_fixed, hist_heur, hist_ql]
strategy_names = ['Fixed High Bitrate', 'Heuristic Buffer-Based', 'Q-Learning Adaptive']
colors = ['#e74c3c', '#f39c12', '#2ecc71']
time_axis = t * STEP_DURATION

for i, (hist, name, color) in enumerate(zip(histories, strategy_names, colors)):
    ax1 = axes[i]
    ax2 = ax1.twinx()
    
    # Bitrate selection
    ax1.step(time_axis, hist['quality'], where='post', color=color, linewidth=2)
    ax1.set_ylabel('Bitrate (Mbps)', color=color)
    ax1.set_ylim(0, 12)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Mark stalls
    stall_times = time_axis[np.array(hist['stall']).astype(bool)]
    for st in stall_times:
        ax1.axvline(x=st, color='red', alpha=0.3, linewidth=3)
    
    # Buffer level
    ax2.plot(time_axis, hist['buffer'], 'gray', alpha=0.4, linewidth=1)
    ax2.set_ylabel('Buffer (s)', color='gray')
    ax2.set_ylim(0, 20)
    
    ax1.set_title(f'{name} | Stalls: {np.sum(hist["stall"]):.0f}', fontsize=13)
    ax1.grid(True, alpha=0.2)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

print("\nDiscussion:")
print("1. Fixed strategy stalls when bandwidth drops below bitrate")
print("2. Heuristic avoids stalls but switches quality too often")
print("3. Q-Learning stays stable - it learned the bandwidth pattern")
print("4. Real systems (YouTube) use heuristics because RL needs retraining per user")


## 实验结果与分析

### 三种策略的典型表现

运行实验后，你通常会观察到以下模式（具体数值受随机种子影响，但趋势稳定）：

- **固定高码率（Fixed）**：平均画质最高（始终 10 Mbps），但卡顿次数明显多于其他两种策略——每当带宽跌破 10 Mbps 且持续一段时间，缓冲区就会被耗尽。QoE 得分通常因卡顿惩罚而大幅降低，即使画质分量最高。
- **启发式规则（Heuristic）**：卡顿次数显著减少，因为它会在缓冲区偏低时主动降低码率「保安全」；但代价是码率频繁在低/中/高档之间跳变，切换次数明显偏高，画质波动感更强。
- **Q-Learning 策略**：通常能同时取得较低的卡顿次数和相对较少的切换次数，QoE 综合得分往往最高，因为它是基于对带宽模式的「先验学习」做出的决策，而非简单的即时规则。

### 现象解读

固定策略的问题在于**完全不感知网络状态**；启发式策略引入了状态感知，但简单的门限规则容易在阈值附近反复横跳（Oscillation）；Q-Learning 由于在训练阶段已经归纳出「什么状态下选什么动作长期收益最高」，决策更平滑。

### 实验局限性

需要注意：**本实验的 Q 表是人工预设的简化版本，并非通过真实强化学习训练过程收敛得到**，仅用于演示 Q-Learning 策略「按状态查表决策」的核心思想。真实系统中的强化学习训练需要大量真实带宽轨迹数据、更精细的状态空间设计（如同时考虑带宽历史趋势），以及针对不同用户/网络环境的持续在线学习或迁移学习。

## 从实验到实际系统

本实验演示了 ABR 决策的核心权衡，但真实流媒体系统更加复杂：

- **YouTube 的 DASH-ABR**：采用基于缓冲区与带宽预测的混合启发式算法（如 BOLA、Dynamic），在数十亿用户规模下追求鲁棒性和可预测性，而非针对单个用户训练强化学习模型；
- **Netflix 的 Pensieve**：是学术界较早将强化学习引入 ABR 决策的系统之一，通过神经网络端到端学习码率选择策略，在实验条件下取得优于传统启发式的效果，但工业落地需要解决泛化性、稳定性、可解释性等问题；
- **实时学习 vs 离线预训练**：本实验的 Q-Learning 策略是离线预训练好的静态策略；真实场景中的强化学习通常需要面对训练数据分布与真实网络环境不匹配（Domain Shift）的挑战，这也是 RL 落地流媒体系统的主要难点之一。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **画质与流畅度存在天然矛盾**：更高的码率意味着更高的卡顿风险，ABR 算法的本质是在两者之间做动态权衡；
2. **QoE 需要综合多个维度**：仅优化画质或仅优化不卡顿都不是好策略，还需考虑码率切换带来的体验损害；
3. **启发式规则简单但存在震荡问题**：基于门限的规则容易在阈值附近反复切换码率；
4. **Q-Learning 体现了「从经验中学习决策」的思想**：通过状态-动作-奖励的迭代，可以学到比手写规则更优的策略；
5. **工业系统的选择是工程权衡的结果**：启发式规则因鲁棒性、可解释性、无需训练而被广泛采用，强化学习则代表了进一步优化的研究方向。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **修改带宽轨迹**：调大 `bandwidth` 生成公式中的噪声幅度（如把 `np.random.normal(0, 1.5, T)` 中的 1.5 改为 4），模拟更不稳定的网络，观察三种策略哪个受影响最大。
2. **调整 Q 表**：手动修改 `Q_table = np.array([0, 0, 1, 2, 2])` 中的数值，尝试让它更「激进」（多选高码率）或更「保守」（多选低码率），观察 QoE 的变化。
3. **增加卡顿惩罚权重**：将 `stall_penalty = total_stall * 3` 中的系数改为 10 或更高，观察这是否会让「启发式」和「Q-Learning」策略之间的差距进一步拉开。
4. **增加第 4 档码率**：在 `QUALITY_LEVELS` 中加入一档更高码率（如 15 Mbps），并相应扩展启发式规则和 Q 表，观察系统整体表现如何变化。
5. **对比带宽均值**：把带宽生成公式中的均值从 8 Mbps 改为 4 Mbps（网络更差）或 12 Mbps（网络更好），比较三种策略的相对优劣是否会发生变化。

---

← [实验一：AI 驱动的网络流量分类](https://www.kaggle.com/code/guopingtan/fmi-demo1-traffic-classification) &nbsp;|&nbsp; 🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验三：视频质量评估 PSNR vs SSIM →](https://www.kaggle.com/code/guopingtan/fmi-demo3-quality-assessment)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University